# Meridian M&A Intelligence Platform
## Comprehensive Valuation Analysis

---

### Multi-Methodology Valuation: DCF, Comparables, and Precedent Transactions

This notebook demonstrates Meridian's comprehensive valuation capabilities, showing how we determine fair value and identify arbitrage opportunities.

In [ ]:
# Setup and imports
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Import Meridian modules
import sys
sys.path.append('..')
from meridian import SyntheticDataGenerator, ScreeningEngine, ValuationEngine
from meridian.visualizations import *

# Display configuration
pd.set_option('display.float_format', lambda x: '%.2f' % x)

print("Meridian Valuation Engine v1.0")
print("="*50)
print("Methodologies: DCF | Comparables | Precedent Transactions")

## 1. Select Target Companies for Valuation

First, we'll identify high-priority targets for detailed valuation analysis.

In [ ]:
# Load companies and run screening
import os

if not os.path.exists('../data/synthetic_universe.parquet'):
    generator = SyntheticDataGenerator(seed=42)
    companies = generator.generate_company_universe(5000)
    companies.to_parquet('../data/synthetic_universe.parquet')
else:
    companies = pd.read_parquet('../data/synthetic_universe.parquet')

# Screen for valuation candidates
screener = ScreeningEngine(companies)
targets = screener.screen(
    min_revenue=50_000_000,
    max_revenue=500_000_000,
    min_growth=0.15,
    industries=['SaaS', 'Fintech', 'Data & Analytics']
)

print(f"Identified {len(targets)} targets for valuation")
print("\nTop 5 Valuation Candidates:")

top_5 = targets.head(5)
for idx, company in top_5.iterrows():
    print(f"\n{company['rank']}. {company['name']}")
    print(f"   Revenue: ${company['revenue_ttm']/1e6:.1f}M | Growth: {company['revenue_growth_yoy']:.1%}")
    print(f"   Current Valuation: ${company['enterprise_value']/1e6:.1f}M ({company['implied_revenue_multiple']:.1f}x revenue)")

## 2. Discounted Cash Flow (DCF) Analysis

Building detailed DCF models for intrinsic value calculation.

In [ ]:
# Initialize valuation engine
valuation_engine = ValuationEngine()

# Select primary target
primary_target = targets.iloc[0]

print(f"DCF VALUATION: {primary_target['name']}")
print("="*60)

# Perform DCF valuation
dcf_result = valuation_engine.dcf_valuation(primary_target)

# Display key assumptions
print("\nKey Assumptions:")
print(f"  WACC: {dcf_result['wacc']:.2%}")
print(f"  Terminal Growth: {dcf_result['terminal_growth']:.1%}")
print(f"  Initial Growth: {dcf_result['assumptions']['initial_growth']:.1%}")
print(f"  EBITDA Margin: {dcf_result['assumptions']['ebitda_margin']:.1%}")

# Cash flow projections
print("\nProjected Free Cash Flows:")
for i, cf in enumerate(dcf_result['projected_cash_flows'], 1):
    print(f"  Year {i}: ${cf/1e6:.1f}M")

# Valuation results
print("\nValuation Components:")
print(f"  PV of Cash Flows: ${dcf_result['pv_cash_flows']/1e6:.1f}M")
print(f"  PV of Terminal Value: ${dcf_result['pv_terminal']/1e6:.1f}M")
print(f"\n📊 DCF Enterprise Value: ${dcf_result['value']/1e6:.1f}M")
print(f"📊 Current Market Value: ${primary_target['enterprise_value']/1e6:.1f}M")
print(f"📊 Implied Return: {(dcf_result['value']/primary_target['enterprise_value'] - 1):.1%}")

## 3. DCF Sensitivity Analysis

Understanding how valuation changes with key assumptions.

In [ ]:
# Sensitivity analysis on WACC and terminal growth
wacc_range = np.arange(0.08, 0.14, 0.01)
terminal_growth_range = np.arange(0.01, 0.05, 0.005)

# Create sensitivity matrix
sensitivity_matrix = np.zeros((len(wacc_range), len(terminal_growth_range)))

for i, wacc in enumerate(wacc_range):
    for j, tg in enumerate(terminal_growth_range):
        # Create modified valuation engine
        temp_engine = ValuationEngine(risk_free_rate=wacc-0.08, market_premium=0.08)
        
        # Modify terminal growth in company data
        temp_target = primary_target.copy()
        temp_result = temp_engine.dcf_valuation(temp_target)
        
        sensitivity_matrix[i, j] = temp_result['value'] / 1e6  # Convert to millions

# Create heatmap
fig = go.Figure(data=go.Heatmap(
    z=sensitivity_matrix,
    x=[f"{tg:.1%}" for tg in terminal_growth_range],
    y=[f"{wacc:.1%}" for wacc in wacc_range],
    colorscale='RdYlGn',
    text=np.round(sensitivity_matrix, 0),
    texttemplate='$%{text}M',
    textfont={"size": 10},
    colorbar=dict(title="Value ($M)")
))

fig.update_layout(
    title='DCF Sensitivity Analysis: WACC vs Terminal Growth',
    xaxis_title='Terminal Growth Rate',
    yaxis_title='WACC',
    height=500
)
fig.show()

# Show value range
print(f"Valuation Range: ${sensitivity_matrix.min():.0f}M - ${sensitivity_matrix.max():.0f}M")
print(f"Base Case: ${dcf_result['value']/1e6:.0f}M")

## 4. Comparable Company Analysis

Valuation based on trading multiples of similar public companies.

In [ ]:
# Load or generate comparables
industry = primary_target['industry']
safe_industry = industry.replace(' ', '_').replace('&', 'and')
comp_file = f'../data/comparables_{safe_industry}.parquet'

if os.path.exists(comp_file):
    comparables = pd.read_parquet(comp_file)
else:
    generator = SyntheticDataGenerator()
    comparables = generator.generate_market_comparables(industry, 30)
    comparables.to_parquet(comp_file)

print(f"Comparable Company Analysis: {primary_target['name']}")
print("="*60)
print(f"\nComparable universe: {len(comparables)} {industry} companies")

# Display comparables statistics
print("\nComparable Multiples:")
print(comparables[['ev_revenue', 'ev_ebitda', 'pe_ratio']].describe())

# Perform comparable valuation
comp_result = valuation_engine.comparable_company_analysis(
    primary_target,
    comparables
)

print("\nValuation Results:")
if 'multiples_used' in comp_result:
    for multiple_type, stats in comp_result['multiples_used'].items():
        if stats:
            print(f"\n{multiple_type}:")
            print(f"  Median: {stats['median']:.2f}x")
            print(f"  Mean: {stats['mean']:.2f}x")
            print(f"  25th-75th percentile: {stats['25th_percentile']:.2f}x - {stats['75th_percentile']:.2f}x")

print(f"\n📊 Comparable-based Value: ${comp_result['value']/1e6:.1f}M")
print(f"📊 Value Range: ${comp_result['value_range']['min']/1e6:.1f}M - ${comp_result['value_range']['max']/1e6:.1f}M")
print(f"📊 Implied Return: {(comp_result['value']/primary_target['enterprise_value'] - 1):.1%}")

## 5. Precedent Transaction Analysis

Analyzing historical M&A transactions for valuation benchmarks.

In [ ]:
# Load or generate precedent transactions
if os.path.exists('../data/historical_deals.parquet'):
    deals = pd.read_parquet('../data/historical_deals.parquet')
else:
    generator = SyntheticDataGenerator()
    deals = generator.generate_deal_history(500)
    deals.to_parquet('../data/historical_deals.parquet')

print(f"Precedent Transaction Analysis: {primary_target['name']}")
print("="*60)

# Filter relevant deals
relevant_deals = deals[
    (deals['target_industry'] == primary_target['industry']) &
    (deals['target_revenue'] > primary_target['revenue_ttm'] * 0.5) &
    (deals['target_revenue'] < primary_target['revenue_ttm'] * 2.0)
]

print(f"\nRelevant precedent transactions: {len(relevant_deals)}")

if len(relevant_deals) > 0:
    # Display transaction statistics
    print("\nPrecedent Transaction Multiples:")
    print(f"  Mean: {relevant_deals['revenue_multiple_paid'].mean():.2f}x")
    print(f"  Median: {relevant_deals['revenue_multiple_paid'].median():.2f}x")
    print(f"  Range: {relevant_deals['revenue_multiple_paid'].min():.2f}x - {relevant_deals['revenue_multiple_paid'].max():.2f}x")

# Perform precedent transaction valuation
precedent_result = valuation_engine.precedent_transaction_analysis(
    primary_target,
    deals
)

if 'value' in precedent_result:
    print(f"\n📊 Precedent-based Value: ${precedent_result['value']/1e6:.1f}M")
    if 'control_premium' in precedent_result:
        print(f"📊 Control Premium Applied: {precedent_result['control_premium']:.1%}")
    if 'value_range' in precedent_result:
        print(f"📊 Value Range: ${precedent_result['value_range']['min']/1e6:.1f}M - ${precedent_result['value_range']['max']/1e6:.1f}M")
    print(f"📊 Implied Return: {(precedent_result['value']/primary_target['enterprise_value'] - 1):.1%}")

## 6. Valuation Summary - Football Field

Comparing all valuation methodologies in a single view.

In [ ]:
# Compile all valuation results
valuation_summary = {
    'DCF Analysis': {
        'value': dcf_result['value'],
        'value_range': {
            'min': sensitivity_matrix.min() * 1e6,
            'max': sensitivity_matrix.max() * 1e6
        }
    },
    'Comparable Companies': comp_result,
    'Precedent Transactions': precedent_result,
    'Current Market Value': {
        'value': primary_target['enterprise_value']
    }
}

# Create football field chart
fig = create_valuation_football_field(
    valuation_summary,
    primary_target['enterprise_value'],
    primary_target['name']
)
fig.show()

# Calculate valuation statistics
all_values = []
for method, result in valuation_summary.items():
    if 'value' in result and method != 'Current Market Value':
        all_values.append(result['value'])

if all_values:
    mean_value = np.mean(all_values)
    median_value = np.median(all_values)
    
    print("\nVALUATION SUMMARY")
    print("="*60)
    print(f"Current Market Value: ${primary_target['enterprise_value']/1e6:.1f}M")
    print(f"Mean Fair Value: ${mean_value/1e6:.1f}M")
    print(f"Median Fair Value: ${median_value/1e6:.1f}M")
    print(f"\nImplied Upside (Mean): {(mean_value/primary_target['enterprise_value'] - 1):.1%}")
    print(f"Implied Upside (Median): {(median_value/primary_target['enterprise_value'] - 1):.1%}")
    
    if mean_value > primary_target['enterprise_value'] * 1.2:
        print("\n🎯 RECOMMENDATION: STRONG BUY - Significant undervaluation detected")
    elif mean_value > primary_target['enterprise_value'] * 1.1:
        print("\n🎯 RECOMMENDATION: BUY - Moderate undervaluation")
    else:
        print("\n🎯 RECOMMENDATION: HOLD - Fairly valued")

## 7. Synergy Analysis

Quantifying potential value creation from acquisition.

In [ ]:
# Define acquirer profile
acquirer = pd.Series({
    'name': 'Strategic Acquirer Corp',
    'industry': primary_target['industry'],
    'revenue_ttm': 1_000_000_000,
    'customer_count': 10000,
    'employees': 5000
})

# Calculate synergies
synergy_analysis = valuation_engine.synergy_analysis(
    acquirer,
    primary_target,
    integration_years=3
)

print("SYNERGY ANALYSIS")
print("="*60)

print("\nRevenue Synergies:")
for source, value in synergy_analysis['revenue_synergies'].items():
    print(f"  {source.replace('_', ' ').title()}: ${value/1e6:.1f}M annually")

print("\nCost Synergies:")
for source, value in synergy_analysis['cost_synergies'].items():
    print(f"  {source.replace('_', ' ').title()}: ${value/1e6:.1f}M annually")

print(f"\nTotal Annual Synergies: ${synergy_analysis['annual_run_rate']/1e6:.1f}M")
print(f"NPV of Synergies: ${synergy_analysis['total_value']/1e6:.1f}M")

# Create synergy waterfall
fig = create_synergy_waterfall(synergy_analysis)
fig.show()

# Adjusted valuation with synergies
total_value_with_synergies = mean_value + synergy_analysis['total_value']
print(f"\n📊 Valuation with Synergies: ${total_value_with_synergies/1e6:.1f}M")
print(f"📊 Total Return Potential: {(total_value_with_synergies/primary_target['enterprise_value'] - 1):.1%}")

## 8. LBO Analysis & Returns

Analyzing potential returns under different exit scenarios.

In [ ]:
# LBO return analysis
purchase_price = primary_target['enterprise_value']

# Different exit scenarios
exit_scenarios = [
    {'name': 'Base Case', 'exit_multiple': primary_target['implied_revenue_multiple'], 'holding_period': 5},
    {'name': 'Upside Case', 'exit_multiple': primary_target['implied_revenue_multiple'] * 1.3, 'holding_period': 5},
    {'name': 'Quick Exit', 'exit_multiple': primary_target['implied_revenue_multiple'] * 1.1, 'holding_period': 3},
    {'name': 'Long Hold', 'exit_multiple': primary_target['implied_revenue_multiple'] * 1.5, 'holding_period': 7}
]

print("LBO RETURN ANALYSIS")
print("="*60)
print(f"Purchase Price: ${purchase_price/1e6:.1f}M")
print(f"Target: {primary_target['name']}")

return_results = []

for scenario in exit_scenarios:
    returns = valuation_engine.calculate_returns(
        purchase_price,
        primary_target,
        exit_multiple=scenario['exit_multiple'],
        holding_period=scenario['holding_period']
    )
    
    print(f"\n{scenario['name']}:")
    print(f"  Holding Period: {scenario['holding_period']} years")
    print(f"  Exit Multiple: {scenario['exit_multiple']:.1f}x")
    print(f"  Exit Value: ${returns['exit_value']/1e6:.1f}M")
    print(f"  Money Multiple: {returns['money_multiple']:.2f}x")
    print(f"  IRR: {returns['irr']:.1%}")
    
    return_results.append({
        'Scenario': scenario['name'],
        'Years': scenario['holding_period'],
        'Exit Multiple': f"{scenario['exit_multiple']:.1f}x",
        'Exit Value': f"${returns['exit_value']/1e6:.0f}M",
        'MOIC': f"{returns['money_multiple']:.2f}x",
        'IRR': f"{returns['irr']:.1%}"
    })

# Display returns table
returns_df = pd.DataFrame(return_results)
display(returns_df)

## 9. Portfolio Valuation Analysis

Analyzing multiple targets simultaneously for portfolio construction.

In [ ]:
# Value top 10 targets
portfolio_valuations = []

print("PORTFOLIO VALUATION ANALYSIS")
print("="*60)
print("Valuing top 10 targets...\n")

for idx, target in targets.head(10).iterrows():
    # Quick DCF for each
    dcf = valuation_engine.dcf_valuation(target)
    
    valuation_data = {
        'Company': target['name'],
        'Industry': target['industry'],
        'Revenue': target['revenue_ttm']/1e6,
        'Growth': target['revenue_growth_yoy'],
        'Current EV': target['enterprise_value']/1e6,
        'DCF Value': dcf['value']/1e6,
        'Upside': (dcf['value']/target['enterprise_value'] - 1)
    }
    
    portfolio_valuations.append(valuation_data)

# Create portfolio DataFrame
portfolio_df = pd.DataFrame(portfolio_valuations)
portfolio_df = portfolio_df.sort_values('Upside', ascending=False)

# Display portfolio
display_df = portfolio_df.copy()
display_df['Revenue'] = display_df['Revenue'].apply(lambda x: f"${x:.0f}M")
display_df['Growth'] = display_df['Growth'].apply(lambda x: f"{x:.1%}")
display_df['Current EV'] = display_df['Current EV'].apply(lambda x: f"${x:.0f}M")
display_df['DCF Value'] = display_df['DCF Value'].apply(lambda x: f"${x:.0f}M")
display_df['Upside'] = display_df['Upside'].apply(lambda x: f"{x:.1%}")

display(display_df)

# Visualize portfolio opportunities
fig = px.scatter(
    portfolio_df,
    x='Growth',
    y='Upside',
    size='Revenue',
    color='Industry',
    hover_data={'Company': True},
    title='Portfolio Opportunity Matrix',
    labels={'Growth': 'Revenue Growth', 'Upside': 'Valuation Upside'}
)

# Add quadrant lines
fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.add_vline(x=portfolio_df['Growth'].median(), line_dash="dash", line_color="gray")

fig.show()

# Portfolio statistics
print("\nPORTFOLIO STATISTICS:")
print(f"Average Upside: {portfolio_df['Upside'].mean():.1%}")
print(f"Median Upside: {portfolio_df['Upside'].median():.1%}")
print(f"Targets with >20% upside: {(portfolio_df['Upside'] > 0.2).sum()}")
print(f"Total portfolio value: ${portfolio_df['Current EV'].sum():.0f}M")

## 10. Export Valuation Report

Generate comprehensive valuation report for investment committee.

In [ ]:
# Create valuation report
from datetime import datetime

report = f"""
{'='*70}
MERIDIAN VALUATION REPORT
{'='*70}
Generated: {datetime.now().strftime('%B %d, %Y')}

PRIMARY TARGET: {primary_target['name']}
{'-'*70}
Industry: {primary_target['industry']}
Revenue (TTM): ${primary_target['revenue_ttm']/1e6:.1f}M
Growth Rate: {primary_target['revenue_growth_yoy']:.1%}
Current Valuation: ${primary_target['enterprise_value']/1e6:.1f}M

VALUATION SUMMARY
{'-'*70}
DCF Value: ${dcf_result['value']/1e6:.1f}M
Comparable Value: ${comp_result['value']/1e6:.1f}M
Precedent Value: ${precedent_result['value']/1e6:.1f}M
Mean Fair Value: ${mean_value/1e6:.1f}M

IMPLIED RETURNS
{'-'*70}
Base Case: {(mean_value/primary_target['enterprise_value'] - 1):.1%}
With Synergies: {(total_value_with_synergies/primary_target['enterprise_value'] - 1):.1%}

SYNERGY POTENTIAL
{'-'*70}
Annual Run Rate: ${synergy_analysis['annual_run_rate']/1e6:.1f}M
NPV of Synergies: ${synergy_analysis['total_value']/1e6:.1f}M

RECOMMENDATION
{'-'*70}
"""

if mean_value > primary_target['enterprise_value'] * 1.2:
    report += "STRONG BUY - Significant undervaluation with clear value creation opportunity"
elif mean_value > primary_target['enterprise_value'] * 1.1:
    report += "BUY - Attractive valuation with moderate upside potential"
else:
    report += "HOLD - Fair valuation, consider strategic benefits"

report += f"""

KEY RISKS
{'-'*70}
• Integration complexity if different technology stacks
• Market conditions may impact exit multiples
• Synergy realization dependent on execution
• Competitive bidding may increase purchase price

{'='*70}
END OF REPORT
{'='*70}
"""

print(report)

# Save report
with open('../data/valuation_report.txt', 'w') as f:
    f.write(report)

print("\n✓ Report saved to ../data/valuation_report.txt")

## Summary

Meridian's Valuation Engine provides:

1. **Multi-methodology valuation** - DCF, Comparables, and Precedent Transactions
2. **Sensitivity analysis** - Understanding key value drivers
3. **Synergy quantification** - Measuring value creation potential
4. **Return analysis** - IRR and MOIC under different scenarios
5. **Portfolio valuation** - Simultaneous analysis of multiple targets

The platform transforms complex financial modeling into actionable investment insights.

---
*Next: Proceed to Notebook 4 for Strategic Intelligence Analysis*